# Tech Challenge - Fase 1
# Sistema Inteligente de Suporte ao Diagnostico Medico

**Pos Tech FIAP - IA para Devs**

| Membro | Responsabilidade |
|--------|------------------|
| Antonio | Exploracao de Dados |
| Renato | Pre-processamento |
| Vinicius Geisler | Modelagem e Avaliacao |
| Marcelo | Visao Computacional (CNN) |
| Vinicius Blasque | Integracao e Caso Clinico |

---

## Contexto do Problema

Um grande hospital universitario precisa de um sistema de IA para apoiar medicos na analise de exames. O volume crescente de pacientes exige uma triagem mais rapida e precisa.

**Objetivo:** Construir um sistema que analise dados de exames (tabulares e imagens) e classifique tumores como **benignos** ou **malignos**, servindo como ferramenta de apoio ao diagnostico.

**Dataset principal:** Breast Cancer Wisconsin — contem medidas computadorizadas de celulas tumorais obtidas por aspiracao por agulha fina (FNA). Sao 569 pacientes com 30 medidas cada.

---

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import warnings, os, joblib, cv2
warnings.filterwarnings('ignore')
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, recall_score, precision_score, f1_score,
                             precision_recall_fscore_support, classification_report,
                             confusion_matrix, roc_curve, auc)
import shap
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
plt.rcParams.update({'figure.figsize': (14, 6), 'font.size': 12})
sns.set_style('whitegrid')
print(f'Bibliotecas carregadas | TensorFlow {tf.__version__}')

---
# PARTE 1: ANALISE EXPLORATORIA DE DADOS (EDA)

A Analise Exploratoria de Dados (EDA) e o primeiro passo em qualquer projeto de Machine Learning. O objetivo e **entender os dados** antes de modelar: verificar a qualidade, identificar padroes, detectar anomalias e formular hipoteses.

### O que fizemos nesta etapa:
- Carregamos o dataset usando `sklearn.datasets.load_breast_cancer()`
- Verificamos a estrutura dos dados com `pandas` (shape, tipos, nulos)
- Analisamos a distribuicao da variavel alvo (benigno vs maligno)
- Criamos visualizacoes com `matplotlib` e `seaborn` para entender as features
- Calculamos a matriz de correlacao para identificar relacoes entre variaveis

---

In [ ]:
cancer = load_breast_cancer()
df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df['diagnostico'] = cancer.target
df['classe'] = df['diagnostico'].map({0: 'Maligno', 1: 'Benigno'})
print(f'Dataset: {df.shape[0]} amostras | {len(cancer.feature_names)} features')
print(f'Valores nulos: {df.isnull().sum().sum()} | Duplicatas: {df.duplicated().sum()}')
df.describe().T

### O que esta tabela mostra:

A funcao `describe()` calcula estatisticas descritivas para cada feature:
- **count**: quantidade de valores nao-nulos (569 em todas = sem dados faltantes)
- **mean**: media dos valores
- **std**: desvio padrao (quanto os valores variam em torno da media)
- **min/max**: valores minimo e maximo
- **25%, 50%, 75%**: quartis da distribuicao

**Observacao importante:** As escalas das features variam muito (ex: mean area vai de 143 a 2501, enquanto mean smoothness vai de 0.05 a 0.16). Isso significa que precisaremos **normalizar** os dados antes da modelagem.

---

In [ ]:
cont = df['classe'].value_counts()
prop = df['classe'].value_counts(normalize=True) * 100
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cores = ['#2ecc71', '#e74c3c']
axes[0].bar(cont.index, cont.values, color=cores, edgecolor='black', alpha=0.85)
for i, v in enumerate(cont.values):
    axes[0].text(i, v+5, f'{v} ({prop.iloc[i]:.1f}%)', ha='center', fontweight='bold', fontsize=13)
axes[0].set_title('Distribuicao do Diagnostico', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Quantidade de Pacientes')
axes[1].pie(cont.values, labels=cont.index, autopct='%1.1f%%', colors=cores, startangle=90, explode=(0.05,0), textprops={'fontsize':13})
axes[1].set_title('Proporcao por Classe', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

### Como analisar este grafico:

O **grafico de barras** (esquerda) mostra a contagem absoluta de cada classe. O **grafico de pizza** (direita) mostra a proporcao.

**O que isso significa:**
- Temos ~63% de casos benignos e ~37% malignos
- Existe um **leve desbalanceamento** — o modelo pode ter tendencia a prever mais "benigno" simplesmente porque e a classe mais comum
- Por isso, nao podemos usar apenas **Acuracia** como metrica. Precisamos monitorar o **Recall (Sensibilidade)** da classe maligno: "de todos os malignos reais, quantos o modelo detectou?"
- Em oncologia, um **falso negativo** (cancer nao detectado) e muito mais grave que um falso positivo (alarme falso)

---

In [ ]:
features_mean = [c for c in df.columns if 'mean' in c]
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i, feat in enumerate(features_mean):
    ax = np.ravel(axes)[i]
    for diag, cor, rotulo in [(0,'#e74c3c','Maligno'), (1,'#2ecc71','Benigno')]:
        ax.hist(df[df['diagnostico']==diag][feat], bins=25, alpha=0.6, color=cor, label=rotulo, edgecolor='black', linewidth=0.5)
    ax.set_title(feat.replace(' ','\n'), fontsize=10, fontweight='bold'); ax.legend(fontsize=7)
plt.suptitle('Distribuicao das Features por Diagnostico', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

### Como analisar os histogramas:

Cada histograma mostra a **distribuicao de valores** de uma feature, separada por classe (vermelho = maligno, verde = benigno). Usamos `matplotlib.pyplot.hist()` com transparencia (`alpha=0.6`) para sobrepor as duas distribuicoes.

**O que procurar:**
- Se as distribuicoes estao **bem separadas** (pouca sobreposicao) → feature discriminativa, util para o modelo
- Se estao **muito sobrepostas** → feature pouco util para distinguir as classes

**Observacoes:**
- `mean radius`, `mean perimeter`, `mean area`: **boa separacao** — tumores malignos sao significativamente maiores
- `mean concavity`, `mean concave points`: **boa separacao** — malignos tem bordas mais irregulares
- `mean smoothness`, `mean fractal dimension`: **muita sobreposicao** — pouco uteis isoladamente

**Conclusao clinica:** Tamanho e irregularidade das bordas sao os melhores indicadores de malignidade, o que e consistente com a literatura medica.

---

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i, feat in enumerate(features_mean):
    ax = np.ravel(axes)[i]
    sns.boxplot(data=df, x='classe', y=feat, ax=ax, palette={'Maligno':'#e74c3c','Benigno':'#2ecc71'}, hue='classe', legend=False)
    ax.set_title(feat.replace(' ','\n'), fontsize=10, fontweight='bold'); ax.set_xlabel(''); ax.set_ylabel('')
plt.suptitle('Boxplots das Features por Diagnostico', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

### Como analisar os boxplots:

O boxplot (criado com `seaborn.boxplot()`) mostra 5 informacoes de cada distribuicao:
- **Linha central**: mediana (valor que divide os dados ao meio)
- **Caixa**: intervalo interquartil (IQR) — contem 50% dos dados (do quartil 25% ao 75%)
- **Bigodes**: estendem-se ate 1.5x IQR alem da caixa
- **Pontos fora dos bigodes**: outliers (valores atipicos)

**O que procurar:** Se as caixas de maligno e benigno **nao se sobrepoe**, a feature separa bem as classes.

**Observacoes:**
- `mean area` e `mean perimeter`: medianas muito diferentes entre classes — excelente separacao
- Malignos tem mais outliers (pontos), indicando maior variabilidade nos tumores malignos

---

In [ ]:
corr_diag = df.drop(columns=['classe']).corr()['diagnostico'].drop('diagnostico').sort_values()
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
cores_b = ['#e74c3c' if v < 0 else '#2ecc71' for v in corr_diag.values]
corr_diag.plot(kind='barh', ax=axes[0], color=cores_b, edgecolor='black', alpha=0.8)
axes[0].set_title('Correlacao das Features com o Diagnostico', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Coeficiente de Correlacao de Pearson'); axes[0].axvline(x=0, color='black', lw=0.8)
df_mean = df[features_mean + ['diagnostico']]
mask = np.triu(np.ones_like(df_mean.corr(), dtype=bool))
sns.heatmap(df_mean.corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0, mask=mask, square=True, ax=axes[1], linewidths=0.5)
axes[1].set_title('Matriz de Correlacao', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

### Como analisar a correlacao:

**Grafico de barras (esquerda):** Mostra a correlacao de Pearson de cada feature com o diagnostico. Calculado com `pandas.DataFrame.corr()`.
- Valores **negativos** (vermelho): quando a feature aumenta, o diagnostico tende a ser maligno (pois maligno=0)
- Valores **positivos** (verde): quando a feature aumenta, tende a ser benigno
- Quanto mais distante de zero, mais forte a relacao

**Heatmap (direita):** Mostra correlacao entre TODAS as features (criado com `seaborn.heatmap()`). Azul = correlacao negativa, vermelho = positiva.

**Descobertas importantes:**
- `worst concave points` (-0.79) e `worst perimeter` (-0.78) sao as mais correlacionadas com malignidade
- `radius`, `perimeter` e `area` sao altamente correlacionadas entre si (~0.99) — **multicolinearidade**. Faz sentido: sao todas medidas de tamanho
- Modelos como Random Forest lidam bem com multicolinearidade; Regressao Logistica pode ser afetada

---
# PARTE 2: PRE-PROCESSAMENTO

O pre-processamento prepara os dados para os algoritmos de Machine Learning. Dados "crus" podem conter escalas diferentes, valores faltantes ou formatos inadequados.

### O que fizemos:
- **Separacao Features/Target**: `X` = 30 features, `y` = diagnostico (0 ou 1)
- **Split estratificado** com `train_test_split()`: 60% treino, 20% validacao, 20% teste
- **Normalizacao** com `StandardScaler`: transforma cada feature para media=0 e desvio padrao=1
- O scaler e ajustado (`fit`) **apenas no treino** para evitar data leakage

---

In [ ]:
X = df[cancer.feature_names]; y = df['diagnostico']
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)
scaler = StandardScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_val_s = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
X_test_s = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)
print('Separacao estratificada (60/20/20):')
for nome, c in [('Treino',y_train),('Validacao',y_val),('Teste',y_test)]:
    p = c.value_counts(normalize=True)
    print(f'  {nome:10s}: {len(c):3d} amostras | Benigno {p.get(1,0)*100:.0f}% | Maligno {p.get(0,0)*100:.0f}%')
print(f'\nApos normalizacao: media={X_train_s.mean().mean():.4f}, desvio={X_train_s.std().mean():.4f}')

### O que os numeros significam:

- **Split estratificado**: `stratify=y` garante que cada conjunto mantenha a mesma proporcao de benigno/maligno do dataset original. Sem isso, um conjunto poderia ter 90% benigno e outro 50%.
- **Normalizacao**: `StandardScaler` aplica a formula `z = (x - media) / desvio_padrao`. Resultado: media ≈ 0 e desvio ≈ 1 para todas as features. Isso e essencial porque:
  - **KNN** calcula distancias — features com valores maiores dominariam
  - **Regressao Logistica** e **SVM** sao sensiveis a escala
  - **Random Forest** nao precisa, mas nao prejudica
- **Data leakage**: Se fizessemos `fit` no dataset inteiro, o scaler "saberia" informacoes do teste, inflando artificialmente as metricas

---
# PARTE 3: MODELAGEM

Treinamos **4 algoritmos diferentes** para comparar qual tem melhor desempenho:

| Modelo | Funcao sklearn | Como funciona |
|--------|---------------|---------------|
| Regressao Logistica | `LogisticRegression()` | Encontra uma fronteira linear que separa as classes |
| Random Forest | `RandomForestClassifier()` | Combina 100 arvores de decisao e vota na classe |
| KNN | `KNeighborsClassifier()` | Classifica com base nos 5 vizinhos mais proximos |
| SVM | `SVC()` | Encontra o hiperplano que maximiza a margem entre classes |

---

In [ ]:
modelos = {
    'Regressao Logistica': LogisticRegression(max_iter=10000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42)
}
resultados = {}; modelos_ok = {}
for nome, m in modelos.items():
    m.fit(X_train_s, y_train); modelos_ok[nome] = m
    yp = m.predict(X_val_s)
    cv = cross_val_score(m, X_train_s, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring='f1_macro')
    resultados[nome] = {'Acuracia': accuracy_score(y_val,yp), 'Sensibilidade (Maligno)': recall_score(y_val,yp,pos_label=0),
        'Precisao (Maligno)': precision_score(y_val,yp,pos_label=0), 'F1 (Maligno)': f1_score(y_val,yp,pos_label=0), 'CV F1 (5-fold)': cv.mean()}
df_res = pd.DataFrame(resultados).T.sort_values('F1 (Maligno)', ascending=False)
df_res

### O que cada metrica significa:

| Metrica | Pergunta que responde | Por que importa |
|---------|----------------------|----------------|
| **Acuracia** | De todos os pacientes, quantos o modelo acertou? | Visao geral, mas enganosa com dados desbalanceados |
| **Sensibilidade (Recall)** | De todos os MALIGNOS reais, quantos foram detectados? | **METRICA PRINCIPAL** — nao podemos deixar cancer passar |
| **Precisao** | De todos que o modelo disse ser maligno, quantos realmente sao? | Evita alarmes falsos excessivos |
| **F1** | Media harmonica entre Precisao e Sensibilidade | Equilibrio entre ambas |
| **CV F1 (5-fold)** | F1 medio em validacao cruzada com 5 divisoes | Robustez — verifica se o resultado nao depende de uma divisao especifica |

---

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
df_res[['Acuracia','Sensibilidade (Maligno)','F1 (Maligno)']].plot(kind='bar', ax=axes[0], colormap='Set2', edgecolor='black', alpha=0.85, rot=15)
axes[0].set_title('Comparacao de Metricas', fontsize=14, fontweight='bold'); axes[0].set_ylabel('Pontuacao'); axes[0].set_ylim(0.8,1.05)
axes[0].axhline(1.0, color='gray', ls='--', alpha=0.3); axes[0].legend(fontsize=8, loc='lower right')
cores_roc = ['#3498db','#2ecc71','#e74c3c','#9b59b6']
for (nome, m), cor in zip(modelos_ok.items(), cores_roc):
    prob = m.predict_proba(X_val_s)[:,1] if hasattr(m,'predict_proba') else m.decision_function(X_val_s)
    fpr, tpr, _ = roc_curve(y_val, prob)
    axes[1].plot(fpr, tpr, color=cor, lw=2, label=f'{nome} (AUC={auc(fpr,tpr):.3f})')
axes[1].plot([0,1],[0,1],'k--',lw=1,label='Aleatorio (0.5)')
axes[1].set_title('Curvas ROC', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Taxa de Falsos Positivos'); axes[1].set_ylabel('Taxa de Verdadeiros Positivos')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
melhor_nome = df_res.index[0]; melhor_modelo = modelos_ok[melhor_nome]
cm = confusion_matrix(y_val, melhor_modelo.predict(X_val_s))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2], xticklabels=['Maligno','Benigno'], yticklabels=['Maligno','Benigno'], linewidths=1, annot_kws={'size':16})
axes[2].set_title(f'Matriz de Confusao\n{melhor_nome}', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Valor Real'); axes[2].set_xlabel('Predicao do Modelo')
plt.tight_layout(); plt.show()

### Como analisar estes graficos:

**Grafico de Barras (esquerda):** Compara as metricas de cada modelo. Quanto mais alta a barra, melhor. A linha tracejada cinza marca 100% (perfeicao).

**Curva ROC (centro):** Mostra a relacao entre verdadeiros positivos (eixo Y) e falsos positivos (eixo X) para diferentes limiares de decisao. Criada com `sklearn.metrics.roc_curve()`.
- **AUC (Area Under Curve)**: area sob a curva. Quanto mais proximo de 1.0, melhor
- Um modelo perfeito tem AUC = 1.0 (curva colada no canto superior esquerdo)
- Um modelo aleatorio tem AUC = 0.5 (linha diagonal tracejada)
- Todos os nossos modelos tem AUC > 0.98, o que e excelente

**Matriz de Confusao (direita):** Mostra acertos e erros do melhor modelo. Criada com `sklearn.metrics.confusion_matrix()` e visualizada com `seaborn.heatmap()`.
- **Canto superior esquerdo**: Maligno correto (Verdadeiro Negativo)
- **Canto inferior direito**: Benigno correto (Verdadeiro Positivo)
- **Canto superior direito**: Maligno predito como Benigno — **ERRO MAIS GRAVE** (falso negativo)
- **Canto inferior esquerdo**: Benigno predito como Maligno (falso positivo — toleravel)

---
# PARTE 4: AVALIACAO FINAL E INTERPRETACAO

Avaliamos o melhor modelo no **conjunto de teste** (dados nunca vistos) e usamos **SHAP** para explicar as decisoes.

---

In [ ]:
y_test_pred = melhor_modelo.predict(X_test_s)
print(f'Melhor modelo: {melhor_nome}\n')
p, r, f, s = precision_recall_fscore_support(y_test, y_test_pred, labels=[0,1])
print(f'{"Classe":>15s} {"Precisao":>10s} {"Sensibilidade":>14s} {"F1":>8s} {"Amostras":>10s}')
print('-'*60)
for i, nc in enumerate(['Maligno','Benigno']): print(f'{nc:>15s} {p[i]:>10.4f} {r[i]:>14.4f} {f[i]:>8.4f} {s[i]:>10d}')
print('-'*60)
print(f'{"Acuracia":>15s} {"":>10s} {"":>14s} {accuracy_score(y_test,y_test_pred):>8.4f} {len(y_test):>10d}')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm_t = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm_t, annot=True, fmt='d', cmap='Blues', ax=axes[0], xticklabels=['Maligno','Benigno'], yticklabels=['Maligno','Benigno'], linewidths=1, annot_kws={'size':18})
axes[0].set_title('Matriz de Confusao (Teste Final)', fontsize=14, fontweight='bold'); axes[0].set_ylabel('Valor Real'); axes[0].set_xlabel('Predicao')
rf = modelos_ok['Random Forest']
imp = pd.Series(rf.feature_importances_, index=cancer.feature_names).sort_values(ascending=True).tail(15)
imp.plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='black', alpha=0.8)
axes[1].set_title('Importancia das Features (Random Forest)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Importancia (Reducao de Impureza Gini)')
plt.tight_layout(); plt.show()

### O que estes resultados significam:

**Tabela de metricas:** Avaliacao final no conjunto de teste (nunca visto durante o treino). Os valores sao as metricas reais do modelo em producao.

**Importancia das Features (direita):** Calculada pela propriedade `feature_importances_` do Random Forest. Cada feature recebe um score baseado em quanto ela reduz a "impureza" (Gini) nas arvores de decisao. Quanto maior a barra, mais a feature contribui para a classificacao.

---

In [ ]:
X_test_np = X_test_s.values; feat_list = X_test_s.columns.tolist()
explainer = shap.TreeExplainer(rf)
sv_raw = explainer.shap_values(X_test_np)
sv = np.array(sv_raw)
if sv.ndim == 3: sv = sv[:,:,0]
elif isinstance(sv_raw, list): sv = np.array(sv_raw[0])
base_val = explainer.expected_value
if isinstance(base_val, (list, np.ndarray)): base_val = np.array(base_val).flatten()[0]
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
plt.sca(axes[0])
shap.summary_plot(sv, X_test_np, feature_names=feat_list, show=False, max_display=15, plot_type='dot')
axes[0].set_title('SHAP: Impacto de Cada Feature', fontsize=13, fontweight='bold'); axes[0].set_xlabel('Valor SHAP (impacto na predicao)')
plt.sca(axes[1])
shap.summary_plot(sv, X_test_np, feature_names=feat_list, show=False, max_display=15, plot_type='bar')
axes[1].set_title('SHAP: Importancia Media', fontsize=13, fontweight='bold'); axes[1].set_xlabel('Valor SHAP medio (absoluto)')
plt.tight_layout(); plt.show()

### Como analisar os graficos SHAP:

SHAP (SHapley Additive exPlanations) e a tecnica mais robusta de explicabilidade em ML. Usa a biblioteca `shap` com `TreeExplainer` para Random Forest.

**Grafico de pontos (esquerda):**
- Cada **ponto** = uma amostra do teste
- **Posicao horizontal** = impacto SHAP (positivo = empurra para maligno, negativo = empurra para benigno)
- **Cor** = valor da feature naquela amostra (vermelho = alto, azul = baixo)
- Exemplo: `worst concave points` alto (vermelho) empurra para a DIREITA (maligno) — isso significa que bordas irregulares indicam cancer

**Grafico de barras (direita):** Media do impacto absoluto de cada feature. Features no topo sao as mais influentes globalmente.

**O que isso representa clinicamente:** O modelo "aprendeu" que tumores malignos sao maiores (worst perimeter) e tem bordas mais irregulares (concave points) — exatamente o que radiologistas procuram.

---
# PARTE 5: VISAO COMPUTACIONAL - CNN (EXTRA)

Alem dos dados tabulares, treinamos uma **Rede Neural Convolucional (CNN)** para analisar imagens de mamografia.

### Tecnologias usadas:
- **TensorFlow/Keras**: framework de deep learning
- **MobileNetV2**: rede pre-treinada no ImageNet (transfer learning)
- **ImageDataGenerator**: augmentation de dados (rotacao, zoom, flip)
- **Grad-CAM**: tecnica de explicabilidade visual que mostra onde a rede "olha"

### Transfer Learning:
Em vez de treinar uma CNN do zero (exigiria milhoes de imagens), usamos o MobileNetV2 que ja aprendeu a reconhecer bordas, texturas e formas. Congelamos esses pesos e treinamos apenas as camadas de classificacao no topo.

---

In [ ]:
BASE_DIR = '../data/images/breast_cancer'
TRAIN_DIR, VAL_DIR, TEST_DIR = [os.path.join(BASE_DIR, s) for s in ['train','val','test']]
IMG_SIZE = (224, 224)
bp = os.path.join(TRAIN_DIR, 'benign')
if not os.path.exists(bp) or len([f for f in os.listdir(bp) if f.endswith('.png')]) == 0:
    print('Gerando dataset sintetico de mamografias para demonstracao...')
    for split in ['train','val','test']:
        for classe in ['benign','malignant']:
            dp = os.path.join(BASE_DIR, split, classe); os.makedirs(dp, exist_ok=True)
            for j in range({'train':80,'val':15,'test':15}[split]):
                img = np.random.randint(20, 80, (224,224,3), dtype=np.uint8)
                cx, cy = np.random.randint(60, 164, 2)
                if classe == 'malignant':
                    cv2.circle(img, (cx,cy), np.random.randint(20,45), (180,180,180), -1)
                    for _ in range(8): cv2.circle(img, (cx+np.random.randint(-25,25),cy+np.random.randint(-25,25)), np.random.randint(5,15), (160,160,160), -1)
                else: cv2.circle(img, (cx,cy), np.random.randint(10,25), (130,130,130), -1)
                cv2.imwrite(os.path.join(dp, f'{classe}_{j:04d}.png'), img)
    print('Pronto!')
train_dg = ImageDataGenerator(rescale=1./255, rotation_range=20, width_shift_range=0.15, height_shift_range=0.15, zoom_range=0.15, horizontal_flip=True, fill_mode='nearest')
val_dg = ImageDataGenerator(rescale=1./255)
train_gen = train_dg.flow_from_directory(TRAIN_DIR, target_size=IMG_SIZE, batch_size=32, class_mode='binary', shuffle=True)
val_gen = val_dg.flow_from_directory(VAL_DIR, target_size=IMG_SIZE, batch_size=32, class_mode='binary', shuffle=False)
test_gen = val_dg.flow_from_directory(TEST_DIR, target_size=IMG_SIZE, batch_size=32, class_mode='binary', shuffle=False)
cls_names = list(train_gen.class_indices.keys())
print(f'Classes: {train_gen.class_indices} | Treino: {train_gen.samples} | Teste: {test_gen.samples}')

In [ ]:
base_m = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_m.trainable = False
x = base_m.output; x = layers.GlobalAveragePooling2D()(x); x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x); x = layers.Dropout(0.2)(x)
out_cnn = layers.Dense(1, activation='sigmoid')(x)
model_cnn = keras.Model(inputs=base_m.input, outputs=out_cnn)
model_cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cbs = [EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
       ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)]
hist = model_cnn.fit(train_gen, epochs=15, validation_data=val_gen, callbacks=cbs, verbose=1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
axes[0].plot(hist.history['accuracy'],'b-',lw=2,label='Treino'); axes[0].plot(hist.history['val_accuracy'],'r-',lw=2,label='Validacao')
axes[0].set_title('Acuracia da CNN', fontweight='bold'); axes[0].set_xlabel('Epoca'); axes[0].set_ylabel('Acuracia'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(hist.history['loss'],'b-',lw=2,label='Treino'); axes[1].plot(hist.history['val_loss'],'r-',lw=2,label='Validacao')
axes[1].set_title('Perda (Loss) da CNN', fontweight='bold'); axes[1].set_xlabel('Epoca'); axes[1].set_ylabel('Perda'); axes[1].legend(); axes[1].grid(alpha=0.3)
test_gen.reset(); y_cnn_prob = model_cnn.predict(test_gen, verbose=0)
y_cnn_pred = (y_cnn_prob > 0.5).astype(int).flatten(); y_cnn_true = test_gen.classes
cm_cnn = confusion_matrix(y_cnn_true, y_cnn_pred)
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues', ax=axes[2], xticklabels=['Benigno','Maligno'], yticklabels=['Benigno','Maligno'], linewidths=1, annot_kws={'size':16})
axes[2].set_title('Matriz de Confusao CNN', fontweight='bold'); axes[2].set_ylabel('Valor Real'); axes[2].set_xlabel('Predicao')
plt.tight_layout(); plt.show()
print(f'Acuracia CNN no teste: {accuracy_score(y_cnn_true, y_cnn_pred):.4f}')

### Como analisar as curvas de treinamento:

**Acuracia (esquerda):** Deve subir ao longo das epocas. Se a curva de treino (azul) sobe mas a de validacao (vermelho) estagna ou cai, temos **overfitting** (modelo decorou o treino mas nao generaliza).

**Perda/Loss (centro):** Deve descer ao longo das epocas. A funcao *binary crossentropy* mede o erro do modelo. Valor mais baixo = modelo mais preciso.

### Como a CNN aprende a distinguir benigno de maligno:

A CNN (Rede Neural Convolucional) aprende **automaticamente** a reconhecer padroes visuais nas imagens, camada por camada:

1. **Camadas iniciais** aprendem padroes simples: bordas, linhas, contrastes
2. **Camadas intermediarias** combinam esses padroes em formas mais complexas: texturas, contornos curvos, regioes densas
3. **Camadas finais** reconhecem padroes de alto nivel: "massa irregular com bordas espiculadas" (tipico de maligno) vs "nodulo redondo e uniforme" (tipico de benigno)

O **MobileNetV2** ja vem pre-treinado com milhoes de imagens do ImageNet e ja sabe extrair bordas, texturas e formas. Nos apenas ensinamos as **camadas de classificacao** (no topo) a usar essas features visuais para decidir: benigno ou maligno.

Durante o treinamento, o modelo viu exemplos de ambas as classes e ajustou seus pesos para **minimizar os erros**. Quando vemos a loss caindo, significa que o modelo esta aprendendo a distinguir as classes cada vez melhor.

---
## Grad-CAM: Como o modelo explica sua decisao visualmente

**Grad-CAM** (Gradient-weighted Class Activation Mapping) e uma tecnica que responde a pergunta: **"onde exatamente na imagem o modelo esta olhando para tomar sua decisao?"**

### Processo tecnico do Grad-CAM:

1. A imagem passa por toda a CNN ate a **ultima camada convolucional** ( do MobileNetV2)
2. Essa camada produz **mapas de ativacao** — cada mapa detecta um padrao diferente (bordas, texturas, formas)
3. Calculamos os **gradientes** da saida (probabilidade de maligno) em relacao a esses mapas de ativacao
4. Os gradientes nos dizem: **"qual mapa de ativacao mais contribuiu para a decisao?"**
5. Fazemos uma media ponderada dos mapas de ativacao usando esses gradientes como pesos
6. O resultado e um **mapa de calor (heatmap)** que mostra as regioes mais importantes

### O que cada cor significa no heatmap:
- **Vermelho/Amarelo** = regiao de ALTA ativacao — o modelo focou aqui para decidir
- **Azul/Frio** = regiao de BAIXA ativacao — o modelo ignorou esta area
- **Retangulo amarelo tracejado** = delimitacao automatica da regiao de maior suspeita (ativacao > 70%)

### O que levou o modelo a classificar como maligno ou benigno:

| Caracteristica Visual | Benigno | Maligno |
|-----------------------|---------|--------|
| Formato do nodulo | Redondo, bem definido | Irregular, espiculado |
| Bordas | Lisas e uniformes | Irregulares com projecoes |
| Tamanho da regiao ativada | Menor, focal | Maior, difusa |
| Intensidade do heatmap | Menos intensa | Muito intensa (vermelho forte) |
| Distribuicao | Concentrada em um ponto | Espalhada com multiplos focos |

Quando o modelo ve uma **massa grande, irregular, com bordas espiculadas**, os filtros convolucionais que detectam "irregularidade" e "bordas nao-uniformes" se ativam fortemente → alta probabilidade de maligno → heatmap vermelho intenso naquela regiao.

Quando o modelo ve um **nodulo pequeno, redondo e uniforme**, os filtros de irregularidade nao se ativam tanto → baixa probabilidade de maligno → heatmap mais frio.

**Por que isso e importante clinicamente:** O medico pode verificar se o modelo esta focando na regiao correta (no nodulo) e nao em artefatos da imagem. Se o heatmap esta vermelho em cima da massa tumoral, isso da **confianca** de que a decisao do modelo e baseada em evidencia visual relevante.

---

In [ ]:
def gradcam(img_arr, modelo, layer='Conv_1'):
    lc = modelo.get_layer(layer)
    gm = tf.keras.models.Model(inputs=modelo.input, outputs=[lc.output, modelo.output])
    with tf.GradientTape() as t: co, pr = gm(img_arr); loss = pr[:,0]
    g = t.gradient(loss, co); w = tf.reduce_mean(g, axis=(0,1,2))
    h = tf.squeeze(co[0] @ w[..., tf.newaxis])
    return (tf.maximum(h, 0) / (tf.math.reduce_max(h) + 1e-8)).numpy()

real_dir = '../data/images/real_samples'; real_imgs = {}
if os.path.exists(real_dir):
    for f in sorted(os.listdir(real_dir)):
        if f.endswith(('.jpg','.png')): real_imgs[f] = cv2.imread(os.path.join(real_dir, f))
if real_imgs:
    n = len(real_imgs); fig, axes = plt.subplots(n, 3, figsize=(16, 5*n))
    if n == 1: axes = axes.reshape(1,-1)
    for idx, (nome, ibgr) in enumerate(real_imgs.items()):
        irgb = cv2.cvtColor(ibgr, cv2.COLOR_BGR2RGB)
        i224 = cv2.resize(irgb, (224,224)).astype(np.float32)/255.0
        batch = np.expand_dims(i224, 0)
        prob = model_cnn.predict(batch, verbose=0)[0][0]
        pred_lbl = 'MALIGNO' if prob > 0.5 else 'BENIGNO'
        conf = prob if prob > 0.5 else 1 - prob
        hm = gradcam(batch, model_cnn); hm_r = cv2.resize(hm, (224,224))
        hm_c = cv2.cvtColor(cv2.applyColorMap(np.uint8(255*hm_r), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)/255.0
        overlay = np.clip(hm_c*0.4 + i224, 0, 1)
        lbl = nome.replace('mamografia_','').replace('.jpg','').replace('_',' ').title()
        axes[idx,0].imshow(irgb); axes[idx,0].set_title(f'Mamografia Real\n({lbl})', fontsize=12, fontweight='bold'); axes[idx,0].axis('off')
        axes[idx,1].imshow(hm_r, cmap='jet'); axes[idx,1].set_title('Mapa de Calor (Grad-CAM)\nRegioes de Atencao', fontsize=12); axes[idx,1].axis('off')
        axes[idx,2].imshow(overlay)
        mask = hm_r > 0.7; coords = np.where(mask)
        if len(coords[0]) > 0:
            from matplotlib.patches import Rectangle
            axes[idx,2].add_patch(Rectangle((coords[1].min(),coords[0].min()), coords[1].max()-coords[1].min(), coords[0].max()-coords[0].min(), lw=2, ec='yellow', fc='none', ls='--'))
            axes[idx,2].text(coords[1].min(), coords[0].min()-3, 'REGIAO SUSPEITA', color='yellow', fontsize=9, fontweight='bold')
        cor = '#e74c3c' if pred_lbl=='MALIGNO' else '#2ecc71'
        axes[idx,2].set_title(f'Analise IA: {pred_lbl} ({conf:.0%})', fontsize=12, fontweight='bold', color=cor); axes[idx,2].axis('off')
    plt.suptitle('Analise de Mamografias Reais com Inteligencia Artificial', fontsize=18, fontweight='bold', y=1.01)
    plt.tight_layout(); plt.show()
    print('Vermelho/Amarelo = alta atencao (possivel nodulo) | Azul = baixa atencao | Retangulo amarelo = regiao suspeita')

### Analise das Mamografias Reais:

As imagens acima sao **mamografias reais** (raio-X de mama) obtidas do Wikimedia Commons (dominio publico, fonte: National Cancer Institute).

Para cada imagem, o sistema realizou:

1. **Pre-processamento**: redimensionamento para 224x224 pixels e normalizacao (valores entre 0 e 1) usando  e divisao por 255
2. **Predicao**: a CNN processou a imagem e retornou uma probabilidade entre 0 (benigno) e 1 (maligno) usando 
3. **Grad-CAM**: calculou o mapa de calor usando  para obter os gradientes da predicao em relacao a camada  do MobileNetV2
4. **Sobreposicao**: o heatmap foi convertido para cores (azul→vermelho) com  e sobreposto na imagem original com transparencia de 40%
5. **Deteccao de regiao suspeita**: pixels com ativacao > 70% do maximo foram marcados com retangulo amarelo

**Como interpretar os resultados:**
- Na mamografia com **cancer (setas)**: o heatmap deve concentrar-se na regiao da massa tumoral visivel (area branca e densa). Isso confirma que o modelo identifica a lesao corretamente.
- Na mamografia **normal vs cancer**: o modelo deve focar mais no lado com a massa (direito) do que no lado normal (esquerdo)
- Na mamografia de **tecido normal**: o heatmap deve ser mais disperso e menos intenso, sem focos vermelhos concentrados

**Importante:** O modelo CNN foi treinado em imagens sinteticas (para demonstracao). Para uso clinico real, seria necessario retreina-lo com o dataset CBIS-DDSM (mamografias reais do Kaggle). Mesmo assim, o Grad-CAM demonstra o **conceito** de como a IA pode ajudar radiologistas a identificar regioes suspeitas.

---
# PARTE 6: CASO CLINICO INTEGRADO

**Cenario:** Paciente mulher, 52 anos, com nodulo palpavel na mama direita detectado em exame de rotina. Realizou aspiracao por agulha fina (FNA). Os resultados das 30 medidas celulares foram processados pelo nosso modelo de Machine Learning.

---

In [ ]:
caso = df[df['diagnostico']==0].iloc[15]
X_caso = caso[cancer.feature_names].values.reshape(1,-1)
X_caso_s = scaler.transform(X_caso)
pred_caso = melhor_modelo.predict(X_caso_s)[0]
probs_caso = melhor_modelo.predict_proba(X_caso_s)[0]
classe_caso = 'MALIGNO' if pred_caso == 0 else 'BENIGNO'
sv_caso_raw = explainer.shap_values(X_caso_s)
sv_caso = np.array(sv_caso_raw)
if sv_caso.ndim == 3: sv_caso = sv_caso[:,:,0]
elif isinstance(sv_caso_raw, list): sv_caso = np.array(sv_caso_raw[0])
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
bars = axes[0].bar(['Maligno','Benigno'], probs_caso, color=['#e74c3c','#2ecc71'], edgecolor='black', alpha=0.85)
for b, v in zip(bars, probs_caso): axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+0.02, f'{v:.1%}', ha='center', fontweight='bold', fontsize=15)
axes[0].set_title(f'Resultado: {classe_caso}', fontsize=16, fontweight='bold', color='#e74c3c' if pred_caso==0 else '#2ecc71')
axes[0].set_ylim(0, 1.15); axes[0].set_ylabel('Probabilidade')
rc=['#2ecc71','#f1c40f','#e67e22','#e74c3c']; rl=['Baixo','Moderado','Alto','Muito Alto']; rr=[0.2,0.5,0.8,1.0]; st=0
for c,l,e in zip(rc,rl,rr): axes[1].barh(0,e-st,left=st,height=0.5,color=c,edgecolor='black',alpha=0.7); axes[1].text((st+e)/2,0,l,ha='center',va='center',fontsize=10,fontweight='bold'); st=e
axes[1].axvline(x=probs_caso[0],color='black',lw=3); axes[1].plot(probs_caso[0],0,'v',color='black',ms=15)
axes[1].text(probs_caso[0],0.35,f'{probs_caso[0]:.0%}',ha='center',fontsize=14,fontweight='bold')
axes[1].set_xlim(0,1); axes[1].set_yticks([]); axes[1].set_title('Escala de Risco', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Probabilidade de Malignidade')
top_idx = np.argsort(np.abs(sv_caso[0]))[-10:]
feats = [cancer.feature_names[i] for i in top_idx]; vals = [sv_caso[0][i] for i in top_idx]
axes[2].barh(feats, vals, color=['#e74c3c' if v>0 else '#3498db' for v in vals], edgecolor='black', alpha=0.8)
axes[2].set_title('SHAP: Por que este diagnostico?', fontsize=14, fontweight='bold')
axes[2].axvline(0, color='black', lw=0.8); axes[2].set_xlabel('Contribuicao (vermelho=maligno, azul=benigno)')
plt.tight_layout(); plt.show()
pm = probs_caso[0]
if pm>0.8: risco,rec = 'ALTO RISCO','Biopsia URGENTE'
elif pm>0.5: risco,rec = 'RISCO MODERADO','Biopsia recomendada'
elif pm>0.2: risco,rec = 'RISCO BAIXO','Acompanhamento em 6 meses'
else: risco,rec = 'RISCO MUITO BAIXO','Acompanhamento de rotina'
print(f'Diagnostico: {classe_caso} | Confianca: {max(probs_caso):.1%} | {risco} | Recomendacao: {rec}')

### Como analisar o caso clinico:

**Grafico de probabilidades (esquerda):** Mostra a probabilidade que o modelo atribui a cada classe. Neste caso, 100% de probabilidade de maligno — o modelo esta muito confiante.

**Escala de risco (centro):** Classifica o resultado em 4 niveis:
- Verde (0-20%): Risco muito baixo — acompanhamento de rotina
- Amarelo (20-50%): Risco moderado — exames complementares
- Laranja (50-80%): Risco alto — biopsia recomendada
- Vermelho (80-100%): Risco muito alto — biopsia urgente

**SHAP individual (direita):** Mostra **por que** o modelo classificou ESTE paciente como maligno. Cada barra representa uma feature:
- **Barras vermelhas** (para direita): empurram para maligno
- **Barras azuis** (para esquerda): empurram para benigno
- As features no topo sao as que mais influenciaram esta decisao especifica

**O que isso representa:** O medico pode ver nao apenas o diagnostico, mas QUAIS medidas do FNA levaram a essa conclusao. Isso da transparencia e confianca para validar ou questionar o resultado.

---
# PARTE 7: DISCUSSAO CRITICA

## O modelo pode ser utilizado na pratica?

**Sim, exclusivamente como ferramenta de APOIO ao diagnostico.**

### Aplicacao proposta:
1. **Triagem automatizada**: Sistema analisa FNA e mamografia, classifica risco
2. **Priorizacao**: Casos de alto risco vao para o topo da fila
3. **Segunda opiniao**: SHAP e Grad-CAM explicam a decisao para o medico
4. **Decisao final**: SEMPRE do medico

### Limitacoes:
| Limitacao | Impacto | Mitigacao |
|-----------|---------|----------|
| Dataset pequeno (569 amostras) | Generalizacao limitada | Validar com dados reais do hospital |
| Nao considera historico do paciente | Pode ignorar fatores de risco | Integrar com prontuario eletronico |
| CNN treinada em dados sinteticos | Nao reflete mamografias reais | Retreinar com CBIS-DDSM do Kaggle |
| Classificacao binaria | Nao distingue subtipos | Expandir para multiclasse |

### Consideracoes Eticas:
- **Transparencia**: Paciente deve ser informado que IA foi usada
- **Responsabilidade**: Decisao final e SEMPRE do medico
- **Privacidade**: Dados devem seguir LGPD
- **Vies**: Modelo treinado em populacao especifica

### **O(a) medico(a) SEMPRE deve ter a palavra final no diagnostico.**

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(melhor_modelo, '../models/melhor_modelo_tabular.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(modelos_ok, '../models/todos_modelos.pkl')
model_cnn.save('../models/modelo_cnn_mama.keras')
print('Modelos salvos em ../models/')